## TOKENIZATION EXPERIMENT

Six different tokenization model have been inspected in order to understand their tokenization behaviour.

The tested models are;
- multilingual e5 large
- gemma 300
- BAAI bge m3
- berturk
- Qwen3 embedding 0.6B
- LaBSE

In [20]:
from transformers import AutoTokenizer,PreTrainedTokenizerBase
from pathlib import Path
import re
import random



In [21]:
para_splitter = "\n\n" 
FRONT_MATTER = re.compile(r"\A---\s*\n.*?\n---\s*\n", re.S)   # block between the first two '---' lines
COMMENT      = re.compile(r"<!--.*?-->", re.S)                 # any HTML comment, also multi-line
BLANK_LINE   = re.compile(r"\n\s*\n")                          # paragraph separator
WHITESPACE   = re.compile(r"\s+")
ONLY_NUMBER  = re.compile(r"\d+")                              # sidebar numbers like '1' ... '16'
MODEL_DICT = {
    "e5" : "intfloat/multilingual-e5-large",
    "gemma" : "google/embeddinggemma-300m",
    "bge" : "BAAI/bge-m3",
    "berturk" : "trmteb/turkish-embedding-model",
    "qwen_06" : "Qwen/Qwen3-Embedding-0.6B",
    "labse" : "sentence-transformers/LaBSE",
    
}

def clean_text(text: str) -> str:
    """Remove front matter and comments; keep the document text and its blank lines."""
    text = FRONT_MATTER.sub("", text, count=1)
    text = COMMENT.sub("", text)
    return text

def split_paragraphs(text: str, keep_headings: bool = False, min_chars: int = 3) -> list[str]:
    """Split cleaned markdown into paragraphs (one string each, whitespace collapsed)."""
    paragraphs = []
    for piece in BLANK_LINE.split(text):
        p = WHITESPACE.sub(" ", piece).strip()
        if not p:
            continue                                  # empty piece
        if p.startswith("#") and not keep_headings:
            continue                                  # markdown heading
        if ONLY_NUMBER.fullmatch(p):
            continue                                  # sidebar number
        if len(p) < min_chars:
            continue                                  # leftover fragment
        paragraphs.append(p)
    return paragraphs

def load_paragraphs(path: Path, **kwargs) -> list[str]:
    text = path.read_text(encoding="utf-8")
    return split_paragraphs(clean_text(text), **kwargs)
def load_paragraphs_text(text) -> list[str]:
    return split_paragraphs(clean_text(text))

In [22]:
para_sample = """Milletimizin kabiliyetini, gençlerimizin dinamizmini, sanayimizin üretim tecrübesini ve teknoloji ekosistemimizin deneyimini dikkate alan yol haritamızla kendi ihtiyaçlarımızı, önceliklerimizi ve değerlerimizi esas alan; insan odaklı, güvenilir ve sürdürülebilir bir yapay zekâ ekosistemi oluşturmayı amaçladık. Bu doğrultuda Eylem Planımızı; “Fark et, İstifade et, Üret, Yönet” olmak üzere 4 temel eksen üzerine inşa ettik. Yapay zekâ ekosistemimizin 5 temel katmanını (enerji, hesaplama-donanım, altyapı, modeller-veri, uygulamalar) ve 3 bileşenini (yetenek, finansman, düzenleme-güven) beslemek üzere 16 eylem belirledik.
"""

In [23]:
para_clean = load_paragraphs_text(para_sample)
print(para_clean)

['Milletimizin kabiliyetini, gençlerimizin dinamizmini, sanayimizin üretim tecrübesini ve teknoloji ekosistemimizin deneyimini dikkate alan yol haritamızla kendi ihtiyaçlarımızı, önceliklerimizi ve değerlerimizi esas alan; insan odaklı, güvenilir ve sürdürülebilir bir yapay zekâ ekosistemi oluşturmayı amaçladık. Bu doğrultuda Eylem Planımızı; “Fark et, İstifade et, Üret, Yönet” olmak üzere 4 temel eksen üzerine inşa ettik. Yapay zekâ ekosistemimizin 5 temel katmanını (enerji, hesaplama-donanım, altyapı, modeller-veri, uygulamalar) ve 3 bileşenini (yetenek, finansman, düzenleme-güven) beslemek üzere 16 eylem belirledik.']


In [24]:
## inspect tokenizer
e5_tokenizer = AutoTokenizer.from_pretrained(MODEL_DICT["e5"])
gemma_tokenizer = AutoTokenizer.from_pretrained(MODEL_DICT["gemma"])
bge_tokenizer = AutoTokenizer.from_pretrained(MODEL_DICT["bge"])
berturk_tokenizer = AutoTokenizer.from_pretrained(MODEL_DICT["berturk"])
qwen_06_tokenizer = AutoTokenizer.from_pretrained(MODEL_DICT["qwen_06"])
labse_tokenizer = AutoTokenizer.from_pretrained(MODEL_DICT["labse"])


In [26]:
e5_pieces = e5_tokenizer.tokenize(para_clean)
gemma_pieces = gemma_tokenizer.tokenize(para_clean)
bge_pieces = bge_tokenizer.tokenize(para_clean)
berturk_pieces = berturk_tokenizer.tokenize(para_clean)
qwen_06_pieces = qwen_06_tokenizer.tokenize(para_clean)
labse_pieces = labse_tokenizer.tokenize(para_clean)

print("model name e5")
print(e5_pieces)
print(len(e5_pieces))
print("*"*100)
print("model name gemma")
print(gemma_pieces)
print(len(gemma_pieces))
print("*"*100)
print("model name bge")
print(bge_pieces)
print(len(bge_pieces))
print("*"*100)
print("model name berturk")
print(berturk_pieces)
print(len(berturk_pieces))
print("*"*100)
print("model name qwen 06")
print(qwen_06_pieces)
print(len(qwen_06_pieces))
print("*"*100)
print("model name labse")
print(labse_pieces)
print(len(labse_pieces))


model name e5
['▁Millet', 'imizin', '▁kabi', 'liye', 'tini', ',', '▁gençler', 'imizin', '▁dinami', 'z', 'mini', ',', '▁sanayi', 'mizin', '▁üretim', '▁tecrübe', 'sini', '▁ve', '▁teknoloji', '▁ekosistem', 'imizin', '▁deneyim', 'ini', '▁dikkat', 'e', '▁alan', '▁yol', '▁har', 'ita', 'mız', 'la', '▁kendi', '▁ihtiyaç', 'larımızı', ',', '▁önce', 'lik', 'lerimizi', '▁ve', '▁değerleri', 'mizi', '▁esas', '▁alan', ';', '▁insan', '▁odaklı', ',', '▁güvenilir', '▁ve', '▁sürdür', 'ül', 'ebilir', '▁bir', '▁ya', 'pay', '▁', 'zek', 'â', '▁ekosistem', 'i', '▁oluştur', 'mayı', '▁amaçla', 'dık', '.', '▁Bu', '▁doğru', 'ltu', 'da', '▁Ey', 'lem', '▁Plan', 'ımızı', ';', '▁“', 'Far', 'k', '▁et', ',', '▁İsti', 'fa', 'de', '▁et', ',', '▁Ü', 'ret', ',', '▁Yön', 'et', '”', '▁olmak', '▁üzere', '▁4', '▁temel', '▁eks', 'en', '▁üzerine', '▁inşa', '▁et', 'tik', '.', '▁Ya', 'pay', '▁', 'zek', 'â', '▁ekosistem', 'imizin', '▁5', '▁temel', '▁kat', 'man', 'ını', '▁(', 'ener', 'ji', ',', '▁hesap', 'lama', '-', 'don', 'an', 'ı